# Exploratory Data Analysis - Perishable Goods Sales Forecasting

This notebook analyzes the sales data for perishable goods to understand patterns, trends, and seasonality.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.data_loader import DataLoader
from config.config import Config

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
# Initialize data loader
loader = DataLoader()
loader.load_data()
loader.preprocess_data()

# Get summary stats
stats = loader.get_summary_stats()
print("\nDataset Summary:")
for key, value in stats.items():
    print(f"{key}: {value}")

## 2. Analyze Perishable Products

In [ ]:
# Get perishable products data
perishable_df = loader.get_perishable_data()

print(f"Perishable products: {Config.PERISHABLE_PRODUCTS}")
print(f"\nTotal records: {len(perishable_df):,}")
print(f"\nSales by product family:")
sales_by_family = perishable_df.groupby('family')['sales'].agg(['sum', 'mean', 'count'])
sales_by_family = sales_by_family.sort_values('sum', ascending=False)
print(sales_by_family)

In [ ]:
# Visualize sales by product family
fig = px.bar(
    sales_by_family.reset_index(),
    x='family',
    y='sum',
    title='Total Sales by Perishable Product Family (2013-2017)',
    labels={'sum': 'Total Sales', 'family': 'Product Family'},
    color='sum',
    color_continuous_scale='Viridis'
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

## 3. Time Series Analysis

In [ ]:
# Daily sales trend for all perishables
daily_sales = perishable_df.groupby('date')['sales'].sum().reset_index()
daily_sales = daily_sales.sort_values('date')

fig = px.line(
    daily_sales,
    x='date',
    y='sales',
    title='Daily Total Sales - All Perishable Products',
    labels={'sales': 'Total Sales', 'date': 'Date'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Sales trend by product family
daily_by_family = perishable_df.groupby(['date', 'family'])['sales'].sum().reset_index()

fig = px.line(
    daily_by_family,
    x='date',
    y='sales',
    color='family',
    title='Daily Sales Trend by Product Family',
    labels={'sales': 'Sales', 'date': 'Date', 'family': 'Product Family'}
)
fig.update_layout(height=600)
fig.show()

## 4. Seasonality Analysis

In [ ]:
# Add time features
perishable_df_time = loader.add_time_features(perishable_df)

# Sales by month
monthly_sales = perishable_df_time.groupby('month')['sales'].mean().reset_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_sales['month_name'] = monthly_sales['month'].apply(lambda x: month_names[x-1])

fig = px.bar(
    monthly_sales,
    x='month_name',
    y='sales',
    title='Average Sales by Month',
    labels={'sales': 'Average Sales', 'month_name': 'Month'},
    color='sales',
    color_continuous_scale='RdYlGn'
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Sales by day of week
dow_sales = perishable_df_time.groupby('dayofweek')['sales'].mean().reset_index()
dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales['day_name'] = dow_sales['dayofweek'].apply(lambda x: dow_names[x])

fig = px.bar(
    dow_sales,
    x='day_name',
    y='sales',
    title='Average Sales by Day of Week',
    labels={'sales': 'Average Sales', 'day_name': 'Day of Week'},
    color='sales',
    color_continuous_scale='Blues'
)
fig.update_layout(height=500)
fig.show()

## 5. Promotion Impact Analysis

In [ ]:
# Compare sales with and without promotions
promo_impact = perishable_df.groupby('onpromotion')['sales'].agg(['mean', 'median', 'sum', 'count'])
promo_impact.index = ['No Promotion', 'On Promotion']
print("\nPromotion Impact:")
print(promo_impact)

# Visualization
fig = go.Figure()
fig.add_trace(go.Bar(
    x=['No Promotion', 'On Promotion'],
    y=promo_impact['mean'],
    name='Average Sales',
    marker_color=['lightblue', 'orange']
))
fig.update_layout(
    title='Average Sales: Promotion vs No Promotion',
    yaxis_title='Average Sales',
    height=500
)
fig.show()

## 6. Holiday Impact Analysis

In [ ]:
# Add holiday features
perishable_with_holidays = loader.add_holiday_features(perishable_df_time)

# Compare sales on holidays vs normal days
holiday_impact = perishable_with_holidays.groupby('is_holiday')['sales'].agg(['mean', 'median', 'sum'])
holiday_impact.index = ['Normal Day', 'Holiday']
print("\nHoliday Impact:")
print(holiday_impact)

## 7. Store Analysis

In [ ]:
# Top 10 stores by sales
top_stores = perishable_df.groupby('store_nbr')['sales'].sum().sort_values(ascending=False).head(10)

fig = px.bar(
    x=top_stores.index.astype(str),
    y=top_stores.values,
    title='Top 10 Stores by Total Perishable Sales',
    labels={'x': 'Store Number', 'y': 'Total Sales'},
    color=top_stores.values,
    color_continuous_scale='Reds'
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Sales by store type
store_type_sales = perishable_df.groupby('type')['sales'].agg(['sum', 'mean', 'count'])
store_type_sales = store_type_sales.sort_values('sum', ascending=False)
print("\nSales by Store Type:")
print(store_type_sales)

## 8. Focus Product Analysis - BREAD/BAKERY

In [ ]:
# Get bread/bakery data (aggregated across all stores)
bread_data = loader.get_product_data('BREAD/BAKERY')
bread_data = bread_data.sort_values('date')

print(f"Bread/Bakery data shape: {bread_data.shape}")
print(f"\nFirst few rows:")
print(bread_data.head())

# Plot
fig = px.line(
    bread_data,
    x='date',
    y='sales',
    title='Daily Sales - BREAD/BAKERY (All Stores)',
    labels={'sales': 'Total Sales', 'date': 'Date'}
)
fig.update_layout(height=500)
fig.show()

## 9. Correlation Analysis

In [ ]:
# Create feature correlation matrix
bread_features = loader.add_time_features(bread_data)
bread_features = loader.create_lag_features(bread_features)
bread_features = bread_features.dropna()

# Select numeric columns for correlation
numeric_cols = bread_features.select_dtypes(include=[np.number]).columns
correlation_cols = [col for col in numeric_cols if col not in ['date']]

corr_matrix = bread_features[correlation_cols].corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix - BREAD/BAKERY')
plt.tight_layout()
plt.show()

# Show correlations with sales
sales_corr = corr_matrix['sales'].sort_values(ascending=False)
print("\nTop features correlated with sales:")
print(sales_corr.head(15))

## 10. Key Insights Summary

Based on the analysis above, document your key findings here:

1. **Sales Patterns**: [Add observations]
2. **Seasonality**: [Add observations]
3. **Promotion Impact**: [Add observations]
4. **Holiday Impact**: [Add observations]
5. **Best Features for Modeling**: [Add observations]